In [ ]:
import pandas as pd

# Chargement du fichier TSV
file_path = '/content/Equus_caballus_NCBI.tsv'
df_equus = pd.read_csv(file_path, sep='\t')

# Affichage des premières lignes
display(df_equus.head())

# Affichage du nombre total de gènes
print(f'Nombre total de gènes : {len(df_equus)}')

: 

In [ ]:
import pandas as pd

# Lecture de la liste des gènes d'intérêt
genes_interest_path = '/content/gene_export.txt'
with open(genes_interest_path, 'r') as f:
    # On suppose un gène par ligne, en nettoyant les espaces/sauts de ligne
    genes_list = [line.strip() for line in f if line.strip()]

# Filtrage du DataFrame
df_filtered = df_equus[df_equus['Symbol'].isin(genes_list)]

# Affichage des résultats
print(f"Nombre de gènes d'intérêt recherchés : {len(set(genes_list))}")
print(f"Nombre de gènes trouvés dans le fichier NCBI : {len(df_filtered)}")
display(df_filtered.head())

In [ ]:
import matplotlib.pyplot as plt

# Calcul des statistiques
total_recherche = len(set(genes_list))
total_trouve = len(df_filtered['Symbol'].unique())
total_perdu = total_recherche - total_trouve

# Calcul des pourcentages
pct_trouve = (total_trouve / total_recherche) * 100
pct_perdu = (total_perdu / total_recherche) * 100

print(f"Match : {pct_trouve:.2f}%")
print(f"Perte : {pct_perdu:.2f}%")

# Création du diagramme circulaire
labels = [f'Match ({total_trouve})', f'Perte ({total_perdu})']
sizes = [total_trouve, total_perdu]
colors = ['#66b3ff', '#ff9999']
explode = (0.1, 0)  # On détache un peu la part 'Match'

plt.figure(figsize=(8, 8))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
plt.title('Pourcentage de match vs perte des gènes d\'intérêt')
plt.axis('equal')  # Pour que le camembert soit bien rond
plt.show()

In [ ]:
# Identification des doublons de symboles dans les résultats filtrés
duplicates = df_filtered[df_filtered.duplicated('Symbol', keep=False)]

print(f"Nombre de lignes en doublon (même symbole) : {len(duplicates)}")
print("Voici quelques exemples de gènes apparaissant plusieurs fois :")
display(duplicates.sort_values('Symbol').head(10))

### Analyse pour Homo sapiens
Nous allons maintenant charger le fichier NCBI pour l'humain et appliquer le même filtrage.

In [ ]:
import pandas as pd

# Chargement du fichier Homo sapiens
file_path_hs = '/content/Homo_sapiens_NCBI.tsv'
df_homo = pd.read_csv(file_path_hs, sep='\t', low_memory=False)

# Affichage des premières lignes
display(df_homo.head())
# Affichage du nombre total de gènes
print(f'Nombre total de gènes : {len(df_homo)}')

In [ ]:
import pandas as pd

# Fusion des données Equus filtrées avec les données Homo sapiens sur le Gene Group Identifier
# On ne garde que les colonnes essentielles pour la clarté
df_matched = pd.merge(
    df_filtered[['Symbol', 'NCBI GeneID', 'Gene Group Identifier']],
    df_homo[['Symbol', 'NCBI GeneID', 'Gene Group Identifier']],
    on='Gene Group Identifier',
    suffixes=('_equus', '_homo')
)

# Suppression des lignes où le Gene Group Identifier est manquant (NaN)
df_matched = df_matched.dropna(subset=['Gene Group Identifier'])

# Affichage du résultat
print(f"Nombre de correspondances trouvées via Gene Group Identifier : {len(df_matched)}")
display(df_matched.head())